<a id="unit-u.d41e8d53c98944d5b3efebf32b06bc14"></a>

<a id="optional-lab-7a-trace-a-real-model-with-litefuse"></a>

# Қосымша 7A зертханалық жұмысы: Нақты модельді Litefuse арқылы трассалау

**7-модуль: Трассаларды талдау және Litefuse** · [Сабақ](course.md) · [7-зертханалық жұмыс](lab07_data_analysis_litefuse.ipynb) · [Тест](quiz07_analysis_and_litefuse.ipynb)

Негізгі жұмыстың 5-бөлімінде модель имитацияланған: дайын жауаптар қайтарып, токен саны мен құнын өзі хабарлаған. Бұл қосымша жұмыс көмекшіге нақты модель беріп, өндірістегі қолданбалардай трассалайды: Langfuse интеграциясы ораған OpenAI SDK әр шақыруды өзі тіркейді — промпт, жауап, модель, параметрлері, модель санаған токендер және ұзақтығы. Кейін нақты шақыруларды имитациялықтармен Doris және Litefuse ішінде салыстырасыз.

Әдепкі модель компьютеріңізде істейді: ашық inference қозғалтқышы llama.cpp сервері және шағын ашық Qwen2.5 0.5B Instruct моделі (Apache 2.0 лицензиясы). CPU қолданады, тіркелгі не кілт керек емес. 3-бөлім балама OpenAI API-ын өз кілтіңізбен қолдануды көрсетеді.

Шамамен 15 минут және алғаш сервер бейнесін (300 MB шамасында), модельді (491 MB) жүктеуге уақыт жоспарлаңыз. Сервер шамамен 0.4 GB жады қолданады. Негізгі жұмыстың 4-бөліміндегі Litefuse және салыстыруға 5-бөлім трассалары қажет. Нақты модель жауаптары, токендері мен уақыты өзгереді, сандарыңыз мысалдан ерекшеленуі мүмкін.

<a id="unit-u.1dbdb508a8334d08993010a76ed0f68c"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.sql(...)` не басқа көмекшілерді қолданбас бұрын осы ұяшықты бір рет орындаңыз.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


<a id="unit-u.6535238969574538b2b1ab2df29e49ea"></a>

<a id="1-check-that-litefuse-runs"></a>

## 1. Litefuse жұмысын тексеру

Келесі ұяшық негізгі жұмыстың 4-бөліміндегідей, қосулы болмаса Litefuse іске қосады; қосулы болса тек тексереді.

In [ ]:
lab.start_litefuse()

<a id="unit-u.68204d7bf8234a63aaf684250781b39f"></a>

**Күтілетін нәтиже**

Кіру адресі мен пайдаланушысы бар «Litefuse is running» карточкасы. Litefuse қосулы болса, порт тексеруі 23100 мәнін `used by this course` деп көрсетеді, ұяшық бірнеше секунд алады.

<a id="unit-u.2dd6c1865d604210bd7d76edc76ce496"></a>

<a id="2-start-the-local-model"></a>

## 2. Жергілікті модельді іске қосу

`lab.start_local_model()` модель файлын Docker томына бір рет жүктеп, SHA-256 бақылау сомасын тексереді. Кейін `environments/llm/compose.llm.yml` файлындағы `doris-obs-llm` llama.cpp серверін 127.0.0.1:28080 (`DORIS_OBS_LLM_PORT`) адресінде қосады. Сервер `/v1` ішінде OpenAI chat completions API-ына жауап береді, кез келген OpenAI клиенті шақыра алады.

Модель Hugging Face ішінен келеді. huggingface.co желіңізден қолжетімсіз болса, Jupyter бастамас бұрын `DORIS_OBS_LLM_MODEL_URL` мәнін айнадағы сол файл адресіне орнатыңыз; бақылау сомасы бірдей екенін көрсетеді. OpenAI қолдансаңыз (3-бөлім), осы ұяшықты өткізіңіз.

In [ ]:
lab.start_local_model()

<a id="unit-u.35d8c1374b8e426697d561b627ae106a"></a>

**Күтілетін нәтиже**

- Порт тексеруі: 28080 (`DORIS_OBS_LLM_PORT`) — `available`, сервер қосулы болса `used by this course`.
- Алғашқыда сервер бейнесі алынған соң әр 5 секундта `Downloaded 240 MB of 491 MB` сияқты жол, кейін бақылау сомасы `/models/qwen2.5-0.5b-instruct-q4_k_m.gguf: OK`. Желіге қарай бір минут не одан көп алады. Кейін `Already downloaded` шығарып, файлды қайта тексереді.
- Сервер іске қосылады, кесте бір `qwen2.5-0.5b-instruct` моделін тізеді.
- Серверден бірнеше секунд кейін «The local model is running» карточкасы: алдымен модельді жадыға жүктейді.

<a id="unit-u.480a4adbb3b64fdea8b4ee4f935d931e"></a>

<a id="3-choose-the-model"></a>

## 3. Модель таңдау

Көмекші модельді OpenAI SDK арқылы шақырады. `MODEL_PROVIDER` шақырулар қайда баратынын шешеді:

- Әдепкі `"local"`: 2-бөлім моделі. llama.cpp сервері API кілтін тексермейді, бірақ SDK талап етеді, сондықтан кез келген мәтін жарайды.
- `"openai"`: `gpt-4o-mini` моделімен OpenAI API-ы. Jupyter бастамас бұрын `OPENAI_API_KEY` айнымалысына кілтіңізді орнатыңыз, мысалы `OPENAI_API_KEY=sk-... .venv/bin/jupyter lab`. Кілтті ноутбукке жазбаңыз: ноутбуктер сақталады, бөлісіледі, commit жасалады. 4-бөлімнің үш сұрағы бір центтен әлдеқайда аз тұрады.

In [ ]:
MODEL_PROVIDER = "local"  # or "openai", with your own key in OPENAI_API_KEY

if MODEL_PROVIDER == "local":
    MODEL_BASE_URL, MODEL_API_KEY, MODEL_NAME = f"{lab.local_model_url}/v1", "not-checked", "qwen2.5-0.5b-instruct"
else:
    # lab.openai_api_key() reads OPENAI_API_KEY, and stops with a card when it is not set.
    MODEL_BASE_URL, MODEL_API_KEY, MODEL_NAME = "https://api.openai.com/v1", lab.openai_api_key(), "gpt-4o-mini"
print(f"The assistant calls {MODEL_NAME} at {MODEL_BASE_URL}")

<a id="unit-u.478d109c627445cb9cc11eac58c03742"></a>

**Күтілетін нәтиже**

`The assistant calls qwen2.5-0.5b-instruct at http://127.0.0.1:28080/v1`, ал `"openai"` болса `gpt-4o-mini at https://api.openai.com/v1`. `"openai"` таңдалған, ортада кілт жоқ болса, «OPENAI_API_KEY is not set» карточкасы орнату жолын айтады; кілттің өзі ешқашан көрсетілмейді.

<a id="unit-u.625717c044184f1ba9fef7906105fec8"></a>

<a id="4-trace-the-assistant-with-the-real-model"></a>

## 4. Нақты модельмен көмекшіні трассалау

Көмекші — негізгі жұмыстың 5-бөліміндегісі, екі айырмасы бар:

- `call_model` жоқ. `answer_question` OpenAI SDK-ның `client.chat.completions.create` әдісін шақырып, бір сұрақ бір жауап алуы үшін `temperature=0` арқылы құжаттардан бір сөйлемдік жауап сұрайды.
- `from langfuse.openai import OpenAI` Langfuse интеграциясы ораған OpenAI SDK клиентін импорттайды. Әр шақыру өзі generation бақылау жазбасына айналады; `name=` арқылы `llm-answer` аталады, орауыш оны сұраудан алып тастайды. Ол хабарларды, жауапты, модельді, параметрлерді, модель жауабындағы токен санын және ұзақтықты тіркейді. Қолмен ештеңе хабарланбайды.

SDK желілік күту шегін 60 секунд, қайталауды ең көбі бір рет қояды. Қайталанған timeout шамамен екі минут және қайталау кідірісін алуы мүмкін. Жауап 128 шығыс токенмен шектелген. Timeout не қосылу қатесі қалпына келтіру нұсқауымен тоқтатады; сәтсіз шақыру Litefuse ішінде көрінеді. 4-бөлімді қайталау жаңа трасса ID-лері тізімін бастайды.

Трассалар `course-assistant-real-model` деп аталады, негізгі жұмыстың `course-assistant` трассаларынан ажырата аласыз. Пайдаланушылар, сеанстар мен сұрақтар сол қалпы.

In [ ]:
import os
import time
from openai import APIConnectionError, APITimeoutError
from doris_course import EnvironmentNotReady

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-doris-course"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-doris-course"
os.environ["LANGFUSE_BASE_URL"] = lab.litefuse_url
os.environ["OTEL_SERVICE_NAME"] = "course-assistant"

from langfuse import get_client, observe, propagate_attributes
from langfuse.openai import OpenAI  # the OpenAI SDK's client, wrapped by Langfuse to trace every call

langfuse = get_client()
client = OpenAI(base_url=MODEL_BASE_URL, api_key=MODEL_API_KEY, timeout=60, max_retries=1)

# The documents that the assistant searches, by topic, as in the Lab.
DOCUMENTS = {
    "doris": "Apache Doris is a real-time analytical database. It answers SQL queries over large tables in seconds.",
    "litefuse": "Litefuse is an LLM engineering platform. It keeps its traces and observations in Apache Doris tables.",
    "collector": "The OpenTelemetry Collector receives logs, metrics, and traces and exports them to Doris.",
}


@observe(as_type="retriever", name="search-docs")
def search_docs(question):
    time.sleep(0.05)  # a lookup in a document index takes a moment
    return [text for topic, text in DOCUMENTS.items() if topic in question.lower()]


@observe(name="answer-question")
def answer_question(question, *, user_id, session_id):
    with propagate_attributes(trace_name="course-assistant-real-model", user_id=user_id, session_id=session_id):
        trace_ids.append(langfuse.get_current_trace_id())
        documents = search_docs(question)
        response = client.chat.completions.create(
            name="llm-answer",  # the observation's name in Litefuse; the wrapper does not send it to the model
            model=MODEL_NAME,
            messages=[
                {"role": "system",
                 "content": "Answer the question in one sentence, with these documents: " + " ".join(documents)},
                {"role": "user", "content": question},
            ],
            temperature=0,
            max_tokens=128,
        )
        return response.choices[0].message.content


trace_ids = []
for user_id, session_id, question in [
    ("user-1", "session-1", "What is Apache Doris?"),
    ("user-1", "session-1", "Where does Litefuse keep its traces?"),
    ("user-2", "session-2", "What is Apache Doris?"),
]:
    try:
        answer = answer_question(question, user_id=user_id, session_id=session_id)
    except (APITimeoutError, APIConnectionError):
        langfuse.flush()
        if MODEL_PROVIDER == "local":
            recovery = f"Check {lab.local_model_url}/health, rerun lab.start_local_model(), then rerun section 4."
        else:
            recovery = "Check the network and MODEL_BASE_URL, then rerun section 4."
        raise EnvironmentNotReady("The model did not answer after at most one retry. " + recovery) from None
    print(f"{user_id}: {question}\n  -> {answer}")
langfuse.flush()
print("Trace IDs:", trace_ids)

<a id="unit-u.4491a409b36149cb80d830320a311510"></a>

**Күтілетін нәтиже**

Үш сұрақ және модель жазған бір сөйлемдік жауаптар, кейін `Trace IDs:` және үш ID. Промпт соны талап еткендіктен, жауаптар құжатқа жақын: мысалы `Apache Doris is a real-time analytical database that answers SQL queries over large tables in seconds.` `temperature=0` кезінде «What is Apache Doris?» (Apache Doris деген не?) сұрағына екі жауап бірдей. Ұяшық бірнеше секунд алады: курс ортасы қатар істеп тұрғанда ноутбук CPU-ында әр шақыру бір-екі секунд.

<a id="unit-u.40b75ff23ccc47e2965f897c7733ef48"></a>

<a id="5-compare-the-real-and-the-simulated-model-calls"></a>

## 5. Нақты және имитациялық модель шақыруларын салыстыру

Келесі ұяшық негізгі жұмыстағыдай Doris ішінде үш трассаның бақылау жазбаларын күтеді.

In [ ]:
TRACE_ID_LIST = ", ".join(f"'{trace_id}'" for trace_id in trace_ids)
real_spans = lab.wait_for_rows(
    f"""
    SELECT trace_id, name, type, provided_model_name AS model, usage_details, total_cost,
           TIMESTAMPDIFF(MICROSECOND, start_time, end_time) / 1000 AS duration_ms
    FROM litefuse.spans_llm_app
    WHERE trace_id IN ({TRACE_ID_LIST})
    ORDER BY start_time, is_root DESC
    """,
    title="The observations of the real model's traces in spans_llm_app",
    hint="Litefuse loads a batch of traces into Doris a few seconds after it receives it. Check the worker's "
         "log with docker logs doris-obs-litefuse-worker, then rerun this cell.",
    min_rows=9,
    timeout_seconds=90,
)

<a id="unit-u.69e93097b50940d8a6e361d63caf15e0"></a>

**Күтілетін нәтиже**

Трассаға үшеуден тоғыз жол. `llm-answer` жолдарында `qwen2.5-0.5b-instruct` моделі, шамамен 1,000–2,000 `duration_ms` және модельден алынған токен сандары бар, мысалы `{"input":50, "output":20, "total":70, "input_cached_tokens":0}`:

- Промпт 26 сөз, бірақ шамамен 50 токен: токендеуші мәтінді көбіне сөзден қысқа бөліктерге бөледі, чат пішімі әр хабар айналасына токен қосады.
- `input_cached_tokens` сервер қайта өңдемей кэштен алған промпт токендерін санайды. llama.cpp сервері өңдеген промпттарын сақтайды; үшінші сұрақ біріншіні қайталайды, сондықтан жаңа серверде шамамен 49 кэштелген, тек 1 жаңа `input` токені бар. OpenAI API-ы да жуырда көрген ұзақ промпттардың кэштелген токендерін хабарлап, оларға төмен баға қолданады.
- `total_cost` — NULL: Litefuse белгілі модельдер тізімінің бағасынан есептейді, компьютеріңіздегі модельге баға жоқ. `gpt-4o-mini` болса, жолда OpenAI бағаларынан Litefuse есептеген құн бар.

<a id="unit-u.134d1c756ef74be3b6b1881255241cd8"></a>

Келесі ұяшық жобаның барлық трассасындағы нақты және негізгі жұмыстың имитациялық шақыруларын модель бойынша салыстырады. Промпт токендері — `total` минус `output`; екі шақыру түрінде де кэштелген токендер кіреді.

In [ ]:
lab.sql("""
SELECT provided_model_name AS model,
       COUNT(*) AS calls,
       ROUND(AVG(usage_details['total'] - usage_details['output'])) AS avg_prompt_tokens,
       ROUND(AVG(usage_details['output'])) AS avg_output_tokens,
       SUM(IFNULL(usage_details['input_cached_tokens'], 0)) AS cached_tokens,
       ROUND(AVG(TIMESTAMPDIFF(MICROSECOND, start_time, end_time)) / 1000) AS avg_duration_ms,
       SUM(total_cost) AS total_cost_usd
FROM litefuse.spans_llm_app
WHERE type = 'GENERATION'
GROUP BY provided_model_name
ORDER BY model
""", title="Model calls by model");

<a id="unit-u.02c4bcfab9a14a3a82e48b73cedfcb2a"></a>

**Күтілетін нәтиже**

Негізгі жұмыстың 5-бөлімі мен осы жұмыстың 4-бөлімін бір рет орындасаңыз, әрқайсысында үш шақыруы бар екі жол:

| `model` | `avg_prompt_tokens` | `avg_output_tokens` | `cached_tokens` | `avg_duration_ms` | `total_cost_usd` |
| --- | --- | --- | --- | --- | --- |
| `qwen2.5-0.5b-instruct` | шамамен 51 | шамамен 18 | шамамен 63 | шамамен 1,000–2,000 | NULL |
| `simulated-llm` | 27 | 16 | 0 | шамамен 165 | `0.002270000000` |

Нақты модель промпт токендерін имитациялық модель санаған сөздерден шамамен екі есе көп санайды, ұзағырақ орындайды және Litefuse білетін құны жоқ. Нақты шақырулар әр орындалуда өзгереді: 4-бөлімді қайталағанда ұзақтық пен кэштелген токендер өзгереді; негізгі жұмыстың имитациялық шақырулары әрдайым сол сандарды береді.

<a id="unit-u.92c5f6193d514a368270e327e7981822"></a>

<a id="6-the-real-model-calls-in-litefuse"></a>

## 6. Нақты модель шақырулары Litefuse ішінде

Litefuse веб-интерфейсінде ([http://127.0.0.1:23100](http://127.0.0.1:23100), негізгі жұмыстың 7-бөліміндегідей кіріп):

1. **Tracing** таңдаңыз. Тізімде негізгі `course-assistant` жанында үш `course-assistant-real-model` трассасы бар.
2. `course-assistant-real-model` трассасын, ағашынан `llm-answer` таңдаңыз. Панель промпт пен жауапты хабар ретінде, `qwen2.5-0.5b-instruct` моделін, орауыш тіркеген `temperature: 0` сияқты параметрлерді, кідіріс пен токендерді көрсетеді. Құны жоқ.
3. **Home** таңдаңыз. Модель диаграммаларында екеуі де бар; generation кідіріс процентильдері нақты шақырулардың қаншалықты ұзағырақ екенін көрсетеді.

<a id="unit-u.253b4013677a45439bcd8b5b5fc63dd4"></a>

<a id="7-try-it"></a>

## 7. Байқап көріңіз

Құжаттар жауап бермейтін «Who created Apache Doris?» (Apache Doris-ті кім жасаған?) сияқты сұрақ қойыңыз. Оны 4-бөлім тізіміне қосып, ұяшықты қайталап, трассасын Litefuse ішінде ашыңыз. `search-docs` Doris құжатын қайтарады, бірақ онда жасаушысы айтылмаған; модель жаттығудан білгенімен жауап береді, мұндай шағын модель жиі қателеседі. Трасса себебін көрсетеді: алған промпт пен берген жауап қатар тұр. LLM трассалары осы үшін: әр сұрауда жауаптың қайдан шыққанын анықтау.

<a id="unit-u.b09666af589c44b2a6b6ca7e525f3775"></a>

<a id="completion-criteria"></a>

## Аяқтау шарттары

- `spans_llm_app` ішінде Langfuse OpenAI интеграциясы өзі тіркеген модель, токен саны және ұзақтығы бар нақты модель трассаларының бақылау жазбалары бар.
- Нақты шақырулар неге сөзден көп токен санайтынын, `input_cached_tokens` нені санайтынын, жергілікті модель құны неге жоқ екенін түсіндіре аласыз.

<a id="clean-up"></a>

## Тазалау

`lab.stop_local_model()` модель серверін тоқтатып, жадын босатады. Модель файлы `doris-obs-llm_models` томында қалады, келесі `lab.start_local_model()` қайта жүктемейді. `lab.teardown()` сервер мен томды қалған ортамен бірге жояды.

<a id="unit-u.caf00a07aac24c48bbf4ce7762784f0b"></a>

<a id="next"></a>

## Әрі қарай

[7-тест: Трассаларды талдау және Litefuse](quiz07_analysis_and_litefuse.ipynb) бөліміне қайтыңыз не [8-модуль: Grafana көмегімен ескерту](../module08-alerting/lab08_alerting.ipynb) бөліміне өтіңіз.